![JupyterLite HIV Drug Resistance Maps: an original conceptual globe, map, and molecular illustration.](assets/hivdr-splash.png)

**[Open this notebook in JupyterLite Lab](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/lab/index.html?path=11_study_evidence_atlas.ipynb)** · **[Read the Jupyter Book](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/book/)** · [Map and globe resources](mapping_resources.md)

Original conceptual artwork; its markers and bars are not observations.

# Study evidence atlas: maps, globe and data gaps

Project: JupyterLite HIV Drug Resistance Maps. Created/updated: 2026-11-16.
Author: [Repository contributors; generated with Codex]. Email/organization: [Not supplied].
Input: factual HIVDB metadata snapshot in `data/studies/`, plus the source-checked annotation ledger.
Output: computed pandas tables and Plotly figures; optional exports under `outputs/`.
Runtime: Python 3.11+, pandas, Plotly 6.2.0, IPython; native retrieval dependencies are listed in `study_query.py`.
Change log: initial reproducible study-inventory notebook. No simulated study data.

**Unit:** a distinct HIVDB reference/publication ID. A publication can describe multiple study components; multiple publications can describe one cohort. This is a descriptive inventory, not a population sample, national resistance estimate, or deduplicated systematic review.

Explore where the selected HIVDB collections contain publication-linked sampling metadata, what information is available, and which studies merit closer extraction. The maps show **evidence availability**, not prevalence, burden, clinical risk, or nationally representative sampling. Geography with no mapped record remains grey; this is not evidence of zero resistance.

In [ ]:
# Read immutable factual inputs and check their recorded hashes before analysis.
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Code, HTML, FileLink
from study_matrix import (load_snapshot, country_membership, build_domains,
                          country_matrix, coverage_table, country_summary,
                          evidence_ledger, DOMAINS, UNKNOWN)
catalogue, evidence, signals, annotations, manifest = load_snapshot()
px.defaults.template = "plotly_white"
print("Snapshot built:", manifest["built_utc"])
print("Catalogue publication records:", len(catalogue))

In [ ]:
# Every view below derives from the same reference-country evidence used in the matrix.
summary = country_summary(catalogue, evidence)
facts = build_domains(catalogue, evidence, annotations)
display(summary.sort_values("Publications", ascending=False).head(12))

## 2D evidence map

Hover for publication counts and the explicit DTG/LEN query subsets. A multinational publication contributes once to each documented country; summing countries double-counts such publications. No country is assigned from author affiliation.

In [ ]:
fig = px.choropleth(summary, locations="iso3", color="Publications", hover_name="country",
    hover_data=["DTG query publications", "LEN query publications", "Publications with exact sample year"],
    color_continuous_scale="Teal", projection="natural earth",
    title="Publications with sampling metadata in selected HIVDB sources")
fig.update_geos(showcoastlines=False, showframe=False, landcolor="#e8edf3")
fig.update_layout(height=540, margin=dict(l=0,r=0,t=55,b=0))
fig.show()

## Interactive globe

Drag to rotate; use the buttons to face Africa, Asia/Pacific or the Americas. This is an orthographic map projection using the same data as the 2D view. Natural Earth provides the public-domain outlines through the pinned Plotly 6.2.0 renderer.

In [ ]:
globe = px.choropleth(summary, locations="iso3", color="Publications", hover_name="country",
    color_continuous_scale="Teal", projection="orthographic", title="Study evidence globe — drag to rotate")
globe.update_geos(projection_rotation=dict(lon=20,lat=-5), showocean=True, oceancolor="#e6f4f8",
    landcolor="#e8edf3", showcoastlines=True, coastlinecolor="white")
globe.update_layout(height=620, margin=dict(l=0,r=0,t=110,b=0), updatemenus=[dict(type="buttons",direction="right",x=0,y=1.08,xanchor="left",yanchor="top",
    buttons=[dict(label=label, method="relayout", args=[{"geo.projection.rotation.lon":lon}])
             for label,lon in [("Africa",20),("Asia/Pacific",120),("Americas",-90)]])])
globe.show()

## DTG and LEN query geography

These counts refer to publications containing selected drug-exposed isolates. They do not cover every study of these drugs, nor every trial site. Missing country metadata can prevent a query publication from appearing on the map. LEN prevention, initial treatment and salvage therapy belong to different clinical questions.

In [ ]:
drug_geo = summary.drop(columns=["Publications"]).melt(id_vars=["iso3","country"], value_vars=["DTG query publications","LEN query publications"],
                         var_name="Query", value_name="Publications")
drug_geo = drug_geo.loc[drug_geo.Publications.gt(0)]
fig = px.choropleth(drug_geo, locations="iso3", color="Publications", hover_name="country", facet_col="Query",
    color_continuous_scale="YlOrRd", title="Country-specific membership in the two explicit exposure queries")
fig.update_geos(showcoastlines=False, showframe=False, landcolor="#e8edf3")
fig.update_layout(height=430, margin=dict(l=0,r=0,t=70,b=0))
fig.show()

## Which domains need extraction?

The bar chart includes the entire selected-source catalogue. “Not reported” means unavailable in this pipeline, not absent from the underlying article. The nine source-checked publications are examples, not a representative sample of study designs.

In [ ]:
coverage = coverage_table(facts, catalogue)
long_coverage = coverage.melt(id_vars="Domain", var_name="Information", value_name="Publications")
fig = px.bar(long_coverage, y="Domain", x="Publications", color="Information", orientation="h",
    color_discrete_map={"With information":"#087e8b","Not reported":"#d9e0e8"},
    title="Metadata coverage across the requested domains", text="Publications")
fig.update_layout(height=470, yaxis=dict(autorange="reversed"), legend=dict(orientation="h",y=1.12))
fig.show()

## Country-by-time evidence map

Exact specimen years only; publication dates and upper bounds are excluded. Each cell counts distinct publications with at least one sampled isolate in that country/year. One study can appear in several years. This is a record-coverage view, not a temporal resistance trend.

In [ ]:
top = summary.nlargest(18,"Publications").country.tolist()
exact = evidence.loc[evidence.collection_year_exact.ne("") & evidence.country.isin(top)]
timeline = exact.pivot_table(index="country",columns="collection_year_exact",values="ref_id",aggfunc="nunique",fill_value=0)
timeline = timeline.reindex(top,fill_value=0)
fig = px.imshow(timeline, color_continuous_scale="Blues", aspect="auto",
    labels=dict(x="Exact collection year",y="Country",color="Publications"),
    title="Where and when are sampling years documented?")
fig.update_layout(height=570)
fig.show()

In [ ]:
# Export interactive graphics directly from Plotly; use the local files for sharing.
out = Path("outputs/study_atlas"); out.mkdir(parents=True,exist_ok=True)
globe.write_html(out / "evidence_globe.html", include_plotlyjs=True)
summary.to_csv(out / "country_evidence_summary.csv", index=False)
print("Saved", (out / "evidence_globe.html").as_posix())

The [mapping resource guide](mapping_resources.md) links to further 2D maps, 3D maps, globes and molecular scenes. Continue to **Meta Analysis** to turn African evidence availability into explicit research questions and a narrowly defined, auditable synthesis.